# Date Dimension Data - Gold Layer

## Objective
Extract and deduplicate unique calendar date attributes from silver.silver_multimodal to build a standardized Date Dimension Gold Delta table (gold.gold_multimodal_dim_date).

## Data Flow
silver.silver_multimodal → Spark SQL / DataFrame → gold.gold_multimodal_dim_date

## Source
The underlying data comes from the multimodal transport network API.

## Input
Silver Delta table: silver.silver_multimodal

## Output
Gold Delta table: gold.gold_multimodal_dim_date

## Gold Layer Principle
The Gold layer delivers curated, dimensional models and business-level aggregations ready for reporting and analytics. This pipeline isolates unique temporal attributes and derives structured date hierarchies to maintain a clean dimension table with strict entity integrity.

## Processing Steps
1. **Load Silver Data:** Read silver.silver_multimodal into PySpark.
2. **Extract Dimension:** Query distinct timestamp values and calculate surrogate date keys, calendar attributes, and weekend indicators.
3. **Write to Gold:** Persist deduplicated dataset to gold.gold_multimodal_dim_date Delta table.

In [0]:
# Load data from silver schema
df_silver_multimodal=spark.table('workspace.silver.silver_multimodal')

In [0]:
# display the dataframe 
df_silver_multimodal.display()

## BUSINESS TRANSFORMATION AND MODELING

In [0]:
# Extract Dimension date Data
query_dim_date = """
SELECT DISTINCT
    CAST(date_format(CAST(timestamp AS TIMESTAMP), 'yyyyMMdd') AS INT) AS date_key,
    CAST(timestamp AS DATE) AS full_date,
    day(CAST(timestamp AS TIMESTAMP)) AS day,
    date_format(CAST(timestamp AS TIMESTAMP), 'EEEE') AS day_name,
    month(CAST(timestamp AS TIMESTAMP)) AS month,
    year(CAST(timestamp AS TIMESTAMP)) AS year,
    CASE WHEN dayofweek(CAST(timestamp AS TIMESTAMP)) IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekend
FROM workspace.silver.silver_multimodal
"""

df_dim_date = spark.sql(query_dim_date)

In [0]:
# Display df_dim_site 
df_dim_date.display()

# WRITING GOLD TABLE

In [0]:
df_dim_date\
    .write\
        .format("delta")\
        .mode("overwrite")\
        .option("overwriteSchema", "true") \
        .saveAsTable("gold.gold_multimodal_dim_date")

# CHECKING THE GOLD TABLE

In [0]:
%sql
SELECT * 
FROM workspace.gold.gold_multimodal_dim_date
LIMIT 10 